# Preparazione dei dataframe per la regressione

Per alleggerire i notebook sulla classificazione, prepariamo separatamente i dataframe. Si organizza questo notebook mantenendo al minimo i commenti. Si contestualizzano quindi i dataframe nel modo in cui vengono utilizzati.

In [1]:
# importiamo i pacchetti necessari
import numpy as np
import pandas as pd

import sys
sys.path.append('../..')

#import src.regr_funcs as fs # ancora non esiste questo file

In [2]:
# importiamo il dataframe sviluppato per l'EDA
dataset_df = pd.read_csv('../../data/processed/dataset_EDA_processed.csv')

# inseriamo in dataset_df una colonna 'day' che contiene il giorno del mese estratto dalla colonna 'date'
dataset_df['date'] = pd.to_datetime(dataset_df['date'])
dataset_df['day'] = (dataset_df["date"] - dataset_df["date"].min()).dt.days + 1

dataset_df.head()

,station_appa,date,hour,CO,NO2,O3,PM10,PM2.5,SO2,geometry_appa,...,temperature,precipitation,winds_spd,winds_dir,geometry_weather,power_area_50,power_area_1000,power_area_2500,week_day,day
0,Borgo Valsugana,2013-11-01,1,NaN,21.0,2.0,19.0,12.0,NaN,POINT (11.45389 46.05184),...,11.350,0.0,NaN,NaN,POINT (11.47747769 46.05804607),17.760273,42.946772,54.704690,friday,1
1,Borgo Valsugana,2013-11-01,2,NaN,18.0,2.0,18.0,11.0,NaN,POINT (11.45389 46.05184),...,11.525,0.0,NaN,NaN,POINT (11.47747769 46.05804607),16.362439,39.762416,52.679229,friday,1
2,Borgo Valsugana,2013-11-01,3,NaN,19.0,2.0,19.0,12.0,NaN,POINT (11.45389 46.05184),...,11.425,0.0,NaN,NaN,POINT (11.47747769 46.05804607),17.861264,41.283915,50.294214,friday,1
3,Borgo Valsugana,2013-11-01,4,NaN,17.0,2.0,20.0,11.0,NaN,POINT (11.45389 46.05184),...,11.075,0.0,NaN,NaN,POINT (11.47747769 46.05804607),14.669913,36.119279,45.105859,friday,1
4,Borgo Valsugana,2013-11-01,5,NaN,18.0,2.0,21.0,13.0,NaN,POINT (11.45389 46.05184),...,10.950,0.0,NaN,NaN,POINT (11.47747769 46.05804607),16.969367,39.626135,48.951222,friday,1


## Regressione oraria

Vogliamo prevedere i valori di ozono, pm10 e pm2.5 per una data ora, a partire dalle ore precedenti a questa. Visto che non tutte le stazioni appa rilevano tutti gli inquinanti, scegliamo di creare tre dataframe separati, uno per ciascun inquinante, in modo da massimizzare il numero di punti contenuti in ciascun dataframe. Ogni riga dei dataframe corrisponde ad una precisa combinazione stazione-giorno-ora.

Usiamo features simili alla classificazione binaria:

- 'station_appa' in formato one-hot: stazione di rilevazione;

- 'elevation': elevazione delle stazioni;

- 'day': processione ordinata di giorni dal primo dato;

- 'cos_week_day': coseno di periodicità del giorno della settimana;

- 'sin_week_day': seno di periodicità del giorno della settimana;

- 'cos_hour': coseno di periodicità dell'ora;

- 'sin_hour': seno di periodicità dell'ora;

- '..._1', '..._2', '..._3': valori per l'inquinante da prevedere relativi a una, due e tre ore prima; 

- '..._diff_1', '..._diff_2': differenze per i valori dell'inquinante da prevedere tra una e due ore prima e tra una e tre ore prima

- 'PM10_1', 'NO2_1': inquinanti e indice un'ora prima;

- 'power_area_50_1', 'power_area_50_2': consumi elettrici entro 50 metri per una, due e tre ore prima;

- 'temperature': temperatura un'ora prima;

- 'precipitation': precipitazioni di 5 ore prima (nell'EDA sembravano quelle correlate maggiormente).

Come 'target' mettiamo il valore dell'inquinante ... all'ora desiderata.

In [3]:
# feature istantanee
hour_temp_df = dataset_df[['station_appa', 'elevation', 'day', 'week_day', 'hour']].copy()
hour_temp_df = pd.get_dummies(hour_temp_df, columns=['station_appa'], prefix='station', dtype=int) # stazioni in one-hot

## scriviamoil giorno della settimana e l'ora come seni e coseni
week_day_map = {'monday':0, 'tuesday':1, 'wednesday':2, 'thursday':3, 'friday':4, 'saturday':5, 'sunday':6}
hour_temp_df['week_day'] = hour_temp_df['week_day'].map(week_day_map) # ritrasformo i giorni della settimana in numeri da 0 a 6

### giorni della settimana
hour_temp_df['cos_week_day'] = np.cos(hour_temp_df['week_day'] * 2*np.pi/7)
hour_temp_df['sin_week_day'] = np.sin(hour_temp_df['week_day'] * 2*np.pi/7)

### ore
hour_temp_df['cos_hour'] = np.cos(hour_temp_df['hour'] * 2*np.pi/24)
hour_temp_df['sin_hour'] = np.sin(hour_temp_df['hour'] * 2*np.pi/24)

## droppiamo le colonne vecchie
hour_temp_df.drop(columns=['hour', 'week_day'], inplace=True)

# riordiniamo le colonne
station_cols = [col for col in hour_temp_df.columns if col.startswith('station_')]
hour_temp_df = hour_temp_df[station_cols + ['elevation', 'day', 'cos_week_day', 'sin_week_day', 'cos_hour', 'sin_hour']]

# aggiungiamo le colonne con il meteo
hour_temp_df['temperature'] = dataset_df.groupby('station_appa')['temperature'].shift(1)
hour_temp_df['precipitation'] = dataset_df.groupby('station_appa')['precipitation'].shift(5)


# DATAFRAME PER L'OZONO

hour_O3_df = hour_temp_df.copy()

# aggiungiamo le colonne con il delay di una, due e tre ore
shift_h = [1, 2, 3]
for h in shift_h:
    hour_O3_df[f'O3_{h}'] = dataset_df.groupby('station_appa')['O3'].shift(h)

# aggiungiamo le colonne con la differenza
hour_O3_df['O3_diff_1'] = dataset_df.groupby('station_appa')['O3'].diff(1).shift(1)
hour_O3_df['O3_diff_2'] = dataset_df.groupby('station_appa')['O3'].diff(2).shift(1)

# aggiungo le colonne con gli altri inquinanti un'ora prima
pol_cols = ['PM10', 'NO2']
hour_O3_df[[col + '_1' for col in pol_cols]] = dataset_df.groupby('station_appa')[pol_cols].shift(1)

# aggiungo le colonne con la potenza una e due ore prima
shift_power_h = [1, 2]
for h in shift_power_h:
    hour_O3_df[f'power_area_50_{h}'] = dataset_df.groupby('station_appa')['power_area_50'].shift(h)

# aggiungiamo il target
hour_O3_df['target'] = dataset_df['O3']

# eliminiamo le righe con NaN. In questo modo cancelliamo sia i NaN creati con shift che quelli delle stazioni che non rivelano l'inquinante desiderato
hour_O3_df.dropna(inplace=True)
hour_O3_df = hour_O3_df.reset_index(drop=True)

# salviamo il dataframe
hour_O3_df.to_csv("../../data/processed/dataset_hour_O3_processed.csv", index=False)


# DATAFRAME PER I PM10

hour_PM10_df = hour_temp_df.copy()

# aggiungiamo le colonne con il delay di una, due e tre ore
shift_h = [1, 2, 3]
for h in shift_h:
    hour_PM10_df[f'PM10_{h}'] = dataset_df.groupby('station_appa')['PM10'].shift(h)

# aggiungiamo le colonne con la differenza
hour_PM10_df['PM10_diff_1'] = dataset_df.groupby('station_appa')['PM10'].diff(1).shift(1)
hour_PM10_df['PM10_diff_2'] = dataset_df.groupby('station_appa')['PM10'].diff(2).shift(1)

# aggiungo le colonne con gli altri inquinanti un'ora prima
pol_cols = ['NO2']
hour_PM10_df[[col + '_1' for col in pol_cols]] = dataset_df.groupby('station_appa')[pol_cols].shift(1)

# aggiungo le colonne con la potenza una e due ore prima
shift_power_h = [1, 2]
for h in shift_power_h:
    hour_PM10_df[f'power_area_50_{h}'] = dataset_df.groupby('station_appa')['power_area_50'].shift(h)

# aggiungiamo il target
hour_PM10_df['target'] = dataset_df['PM10']

# eliminiamo le righe con NaN. In questo modo cancelliamo sia i NaN creati con shift che quelli delle stazioni che non rivelano l'inquinante desiderato
hour_PM10_df.dropna(inplace=True)
hour_PM10_df = hour_PM10_df.reset_index(drop=True)

# salviamo il dataframe
hour_PM10_df.to_csv("../../data/processed/dataset_hour_PM10_processed.csv", index=False)


# DATAFRAME PER I PM2.5

hour_PM2_5_df = hour_temp_df.copy()

# aggiungiamo le colonne con il delay di una, due e tre ore
shift_h = [1, 2, 3]
for h in shift_h:
    hour_PM2_5_df[f'PM2.5_{h}'] = dataset_df.groupby('station_appa')['PM2.5'].shift(h)

# aggiungiamo le colonne con la differenza
hour_PM2_5_df['PM2.5_diff_1'] = dataset_df.groupby('station_appa')['PM2.5'].diff(1).shift(1)
hour_PM2_5_df['PM2.5_diff_2'] = dataset_df.groupby('station_appa')['PM2.5'].diff(2).shift(1)

# aggiungo le colonne con gli altri inquinanti un'ora prima
pol_cols = ['NO2']
hour_PM2_5_df[[col + '_1' for col in pol_cols]] = dataset_df.groupby('station_appa')[pol_cols].shift(1)

# aggiungo le colonne con la potenza una e due ore prima
shift_power_h = [1, 2]
for h in shift_power_h:
    hour_PM2_5_df[f'power_area_50_{h}'] = dataset_df.groupby('station_appa')['power_area_50'].shift(h)

# aggiungiamo il target
hour_PM2_5_df['target'] = dataset_df['PM2.5']

# eliminiamo le righe con NaN. In questo modo cancelliamo sia i NaN creati con shift che quelli delle stazioni che non rivelano l'inquinante desiderato
hour_PM2_5_df.dropna(inplace=True)
hour_PM2_5_df = hour_PM2_5_df.reset_index(drop=True)

# salviamo il dataframe
hour_PM2_5_df.to_csv("../../data/processed/dataset_hour_PM2_5_processed.csv", index=False)

## Regressione giornaliera

Vogliamo prevedere i valori di ozono, pm10 e pm2.5 per un dato giorno, a partire dai giorni precedenti. Visto che non tutte le stazioni appa rilevano tutti gli inquinanti, scegliamo di creare tre dataframe separati, uno per ciascun inquinante, in modo da massimizzare il numero di punti contenuti in ciascun dataframe. Ogni riga dei dataframe corrisponde ad una precisa combinazione stazione-giorno.

Usiamo features simili alla classificazione binaria:

- 'station_appa' in formato one-hot: stazione di rilevazione;

- 'elevation': elevazione delle stazioni;

- 'day': processione ordinata di giorni dal primo dato;

- 'cos_week_day': coseno di periodicità del giorno della settimana;

- 'sin_week_day': seno di periodicità del giorno della settimana;

- '..._1', '..._2': valori medi per la concentrazione dell'inquinante da prevedere relativi a uno e due giorni prima; 

- '..._diff': differenza per i valori medi dell'inquinante da prevedere tra uno e due giorni prima

- 'PM10_1', 'NO2_1': inquinanti medi un giorno prima;

- 'power_area_50_1': consumo elettrico medio entro 50 metri per il giorno precedente

- 'temperature': temperatura media del giorno prima;

- 'precipitation': precipitazioni medie del giorno precedente

Come 'target' mettiamo il valore dell'inquinante ... relativo al giorno desiderato.

In [4]:
# costruiamo il dataframe accorpando i dati delle ore diverse
temp_df = (dataset_df.groupby(['station_appa', 'day'], as_index=False).agg(
    elevation=('elevation', 'first'),
    week_day=('week_day', 'first'),
    PM10=('PM10', 'mean'),
    PM2_5=('PM2.5', 'mean'),
    O3=('O3', 'mean'),
    NO2=('NO2', 'mean'),
    power_area_50=('power_area_50', 'mean'),
    temperature=('temperature', 'mean'),
    precipitation=('precipitation', 'mean')))

temp_df = temp_df.rename(columns={'PM2_5': 'PM2.5'})


# feature istantanee
day_temp_df = temp_df[['station_appa', 'elevation', 'day', 'week_day']].copy()
day_temp_df = pd.get_dummies(day_temp_df, columns=['station_appa'], prefix='station', dtype=int) # stazioni in one-hot

## scriviamoil giorno della settimana e l'ora come seni e coseni
week_day_map = {'monday':0, 'tuesday':1, 'wednesday':2, 'thursday':3, 'friday':4, 'saturday':5, 'sunday':6}
day_temp_df['week_day'] = day_temp_df['week_day'].map(week_day_map) # ritrasformo i giorni della settimana in numeri da 0 a 6

### giorni della settimana
day_temp_df['cos_week_day'] = np.cos(day_temp_df['week_day'] * 2*np.pi/7)
day_temp_df['sin_week_day'] = np.sin(day_temp_df['week_day'] * 2*np.pi/7)

## droppiamo le colonne vecchie
day_temp_df.drop(columns=['week_day'], inplace=True)

# riordiniamo le colonne
station_cols = [col for col in day_temp_df.columns if col.startswith('station_')]
day_temp_df = day_temp_df[station_cols + ['elevation', 'day', 'cos_week_day', 'sin_week_day']]

# aggiungiamo le colonne con il meteo
day_temp_df['temperature'] = temp_df.groupby('station_appa')['temperature'].shift(1)
day_temp_df['precipitation'] = temp_df.groupby('station_appa')['precipitation'].shift(1)


# DATAFRAME PER L'OZONO

day_O3_df = day_temp_df.copy()

# aggiungiamo le colonne con il delay di uno e due giorni
shift_d = [1, 2]
for d in shift_d:
    day_O3_df[f'O3_{d}'] = temp_df.groupby('station_appa')['O3'].shift(d)

# aggiungiamo la colonna con la differenza
day_O3_df['O3_diff_1'] = temp_df.groupby('station_appa')['O3'].diff(1).shift(1)

# aggiungiamo le colonne con gli altri inquinanti un giorno prima
pol_cols = ['PM10', 'NO2']
day_O3_df[[col + '_1' for col in pol_cols]] = temp_df.groupby('station_appa')[pol_cols].shift(1)

# aggiungo le colonne con la potenza un giorno prima
day_O3_df['power_area_50_1'] = temp_df.groupby('station_appa')['power_area_50'].shift(1)

# aggiungiamo il target
day_O3_df['target'] = temp_df['O3']

# eliminiamo le righe con NaN. In questo modo cancelliamo sia i NaN creati con shift che quelli delle stazioni che non rivelano l'inquinante desiderato
day_O3_df.dropna(inplace=True)
day_O3_df = day_O3_df.reset_index(drop=True)

# salviamo il dataframe
day_O3_df.to_csv("../../data/processed/dataset_day_O3_processed.csv", index=False)


# DATAFRAME PER I PM10

day_PM10_df = day_temp_df.copy()

# aggiungiamo le colonne con il delay di uno e due giorni
shift_d = [1, 2]
for d in shift_d:
    day_PM10_df[f'PM10_{d}'] = temp_df.groupby('station_appa')['PM10'].shift(d)

# aggiungiamo la colonna con la differenza
day_PM10_df['PM10_diff_1'] = temp_df.groupby('station_appa')['PM10'].diff(1).shift(1)

# aggiungo le colonne con gli altri inquinanti un giorno prima
pol_cols = ['NO2']
day_PM10_df[[col + '_1' for col in pol_cols]] = temp_df.groupby('station_appa')[pol_cols].shift(1)

# aggiungo le colonne con la potenza un giorno prima
day_PM10_df['power_area_50_1'] = temp_df.groupby('station_appa')['power_area_50'].shift(1)

# aggiungiamo il target
day_PM10_df['target'] = temp_df['PM10']

# eliminiamo le righe con NaN. In questo modo cancelliamo sia i NaN creati con shift che quelli delle stazioni che non rivelano l'inquinante desiderato
day_PM10_df.dropna(inplace=True)
day_PM10_df = day_PM10_df.reset_index(drop=True)

# salviamo il dataframe
day_PM10_df.to_csv("../../data/processed/dataset_day_PM10_processed.csv", index=False)


# DATAFRAME PER I PM2.5

day_PM2_5_df = day_temp_df.copy()

# aggiungiamo le colonne con il delay di uno e due giorni
shift_d = [1, 2]
for d in shift_d:
    day_PM2_5_df[f'PM2.5_{d}'] = temp_df.groupby('station_appa')['PM2.5'].shift(d)

# aggiungiamo la colonna con la differenza
day_PM2_5_df['PM2.5_diff_1'] = temp_df.groupby('station_appa')['PM2.5'].diff(1).shift(1)

# aggiungiamo le colonne con gli altri inquinanti un giorno prima
pol_cols = ['NO2']
day_PM2_5_df[[col + '_1' for col in pol_cols]] = temp_df.groupby('station_appa')[pol_cols].shift(1)

# aggiungiamo la colonna con la potenza un giorno prima
day_PM2_5_df['power_area_50_1'] = temp_df.groupby('station_appa')['power_area_50'].shift(1)

# aggiungiamo il target
day_PM2_5_df['target'] = temp_df['PM2.5']

# eliminiamo le righe con NaN. In questo modo cancelliamo sia i NaN creati con shift che quelli delle stazioni che non rivelano l'inquinante desiderato
day_PM2_5_df.dropna(inplace=True)
day_PM2_5_df = day_PM2_5_df.reset_index(drop=True)

# salviamo il dataframe
day_PM2_5_df.to_csv("../../data/processed/dataset_day_PM2_5_processed.csv", index=False)